# Day 3 stretch. The Trainer and a LoRA adapter

SetFit got everyone a working model. This optional notebook shows the same left-right task the
way a full pipeline does it, with the Hugging Face `Trainer` and a parameter-efficient LoRA
adapter. It is a guided demo, not something you have to type live.

## Load the workshop helpers

One line fetches the helper functions we use across the workshop. Open
`workshop_utils.py` in the file browser on the left if you want to read them.

In [ ]:
!wget -q -O workshop_utils.py https://raw.githubusercontent.com/jacqpark/instats-python-workshop/main/workshop_utils.py
from workshop_utils import pull_manifesto
print('helpers loaded')

## What LoRA does

Fine-tuning every weight in a model is expensive. LoRA freezes the big pretrained model and trains
two small adapter matrices next to each attention layer. You end up training well under one
percent of the weights, which fits a free GPU and produces a tiny file you can share.

In [ ]:
!pip install -q peft

## Data and tokenizer

Same corpus and tokenizer as Day 2. Pull, collapse to left and right, tokenize into a Dataset.

In [ ]:
from google.colab import userdata
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

df = pull_manifesto(userdata.get('MANIFESTO_KEY'))
df = df[df['rile'].notna()].reset_index(drop=True)
# One country keeps the vocabulary consistent, which is what a small
# training set needs. All four at once costs about 10 kappa points.
df = df[df['countryname'] == 'United Kingdom'].reset_index(drop=True)
df['labels'] = (df['rile'] == 'right').astype(int)

MODEL = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
def tok(b): return tokenizer(b['text'], truncation=True, padding='max_length', max_length=128)
ds = Dataset.from_pandas(df[['text','labels']]).map(tok, batched=True)
ds = ds.train_test_split(test_size=0.2, seed=42)
print(ds)

## Wrap the model with LoRA

Load the encoder with a classification head, then wrap it in a LoRA config. Print the trainable
count to see how small the adapter is.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                  target_modules=['q_lin','v_lin'])
model = get_peft_model(base, lora)
model.print_trainable_parameters()

## Train with the Hugging Face Trainer

The `Trainer` runs the loop from Day 3. A couple of epochs on a free GPU is enough for this task.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, cohen_kappa_score

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'kappa': cohen_kappa_score(p.label_ids, pred),
            'f1': f1_score(p.label_ids, pred, average='macro')}

args = TrainingArguments(output_dir='lora_out', per_device_train_batch_size=16,
                         num_train_epochs=2, learning_rate=2e-4, logging_steps=20,
                         report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=ds['train'],
                  eval_dataset=ds['test'], compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

## Save just the adapter

The LoRA adapter comes to just under 3 MB, small enough to email, while the full model runs to
hundreds of megabytes.

In [ ]:
model.save_pretrained('left_right_lora')
import os
mb = sum(os.path.getsize(os.path.join('left_right_lora', f)) for f in os.listdir('left_right_lora')) / 1e6
print(f'adapter size on disk: {mb:.2f} MB')

## The generative cousin

You fine-tuned an encoder to classify. The same LoRA idea fine-tunes a generative model to write
or follow instructions, on the same free GPU. The pipeline you learned this week is the on-ramp to
that. Where you take it next is up to your research question.